<a href="https://colab.research.google.com/github/Janbidhiman-04/projectclassrom/blob/main/Notes_created.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q gradio
!pip install -q pandas
!pip install -q bcrypt
!pip install -q qrcode[pil]
!pip install -q plotly
!pip install -q python-multipart
!pip install -q email-validator
!pip install -q faker

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = "/content/drive/MyDrive/ClassroomSystem"
os.makedirs(BASE_PATH, exist_ok=True)

print(f"✅ Data will be stored at: {BASE_PATH}")

Mounted at /content/drive
✅ Data will be stored at: /content/drive/MyDrive/ClassroomSystem


In [7]:
# Import all required libraries and setup database

import os
import json
import uuid
import random
import string
import bcrypt
import datetime
import qrcode
from datetime import datetime, timedelta
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
from PIL import Image
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML, Image as IPImage

# Database class for storing all data
class Database:
    def __init__(self, base_path):
        self.base_path = base_path

        # Define all data files
        self.users_file = os.path.join(base_path, 'users.json')
        self.classes_file = os.path.join(base_path, 'classes.json')
        self.enrollments_file = os.path.join(base_path, 'enrollments.json')
        self.assignments_file = os.path.join(base_path, 'assignments.json')
        self.submissions_file = os.path.join(base_path, 'submissions.json')
        self.grades_file = os.path.join(base_path, 'grades.json')
        self.announcements_file = os.path.join(base_path, 'announcements.json')

        # Initialize all files
        self.init_files()

    def init_files(self):
        files = [self.users_file, self.classes_file, self.enrollments_file,
                self.assignments_file, self.submissions_file, self.grades_file,
                self.announcements_file]

        for file in files:
            if not os.path.exists(file):
                with open(file, 'w') as f:
                    json.dump([], f)

    def read_data(self, file_path):
        with open(file_path, 'r') as f:
            return json.load(f)

    def write_data(self, file_path, data):
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2, default=str)

    # User operations
    def get_users(self):
        return self.read_data(self.users_file)

    def save_user(self, user):
        users = self.get_users()
        users.append(user)
        self.write_data(self.users_file, users)

    def find_user_by_email(self, email):
        users = self.get_users()
        for user in users:
            if user['email'].lower() == email.lower():
                return user
        return None

    def find_user_by_id(self, user_id):
        users = self.get_users()
        for user in users:
            if user['id'] == user_id:
                return user
        return None

    # Class operations
    def get_classes(self):
        return self.read_data(self.classes_file)

    def save_class(self, class_data):
        classes = self.get_classes()
        classes.append(class_data)
        self.write_data(self.classes_file, classes)

    def find_class_by_code(self, code):
        classes = self.get_classes()
        for cls in classes:
            if cls['invite_code'] == code:
                return cls
        return None

    def find_class_by_id(self, class_id):
        classes = self.get_classes()
        for cls in classes:
            if cls['id'] == class_id:
                return cls
        return None

    def get_teacher_classes(self, teacher_id):
        classes = self.get_classes()
        return [c for c in classes if c['teacher_id'] == teacher_id]

    # Enrollment operations
    def get_enrollments(self):
        return self.read_data(self.enrollments_file)

    def add_enrollment(self, enrollment):
        enrollments = self.get_enrollments()
        enrollments.append(enrollment)
        self.write_data(self.enrollments_file, enrollments)

    def get_student_classes(self, student_id):
        enrollments = self.get_enrollments()
        class_ids = [e['class_id'] for e in enrollments if e['user_id'] == student_id]
        classes = self.get_classes()
        return [c for c in classes if c['id'] in class_ids]

    def is_enrolled(self, user_id, class_id):
        enrollments = self.get_enrollments()
        for e in enrollments:
            if e['user_id'] == user_id and e['class_id'] == class_id:
                return True
        return False

    # Assignment operations
    def get_assignments(self):
        return self.read_data(self.assignments_file)

    def save_assignment(self, assignment):
        assignments = self.get_assignments()
        assignments.append(assignment)
        self.write_data(self.assignments_file, assignments)

    def get_class_assignments(self, class_id):
        assignments = self.get_assignments()
        return [a for a in assignments if a['class_id'] == class_id]

    # Submission operations
    def get_submissions(self):
        return self.read_data(self.submissions_file)

    def add_submission(self, submission):
        submissions = self.get_submissions()
        submissions.append(submission)
        self.write_data(self.submissions_file, submissions)

    def get_student_submission(self, assignment_id, student_id):
        submissions = self.get_submissions()
        for sub in submissions:
            if sub['assignment_id'] == assignment_id and sub['student_id'] == student_id:
                return sub
        return None

    # Grade operations
    def get_grades(self):
        return self.read_data(self.grades_file)

    def save_grade(self, grade):
        grades = self.get_grades()
        # Update if exists
        for i, g in enumerate(grades):
            if g['assignment_id'] == grade['assignment_id'] and g['student_id'] == grade['student_id']:
                grades[i] = grade
                self.write_data(self.grades_file, grades)
                return
        grades.append(grade)
        self.write_data(self.grades_file, grades)

    # Announcement operations
    def get_announcements(self):
        return self.read_data(self.announcements_file)

    def add_announcement(self, announcement):
        announcements = self.get_announcements()
        announcements.append(announcement)
        self.write_data(self.announcements_file, announcements)

    def get_class_announcements(self, class_id):
        announcements = self.get_announcements()
        return [a for a in announcements if a['class_id'] == class_id]

# Initialize database
db = Database(BASE_PATH)
print("✅ Database initialized!")
print(f"📁 Data files created in: {BASE_PATH}")

✅ Database initialized!
📁 Data files created in: /content/drive/MyDrive/ClassroomSystem


In [8]:
# CELL 4: Main Classroom Manager Class

import bcrypt
import secrets

class ClassroomManager:
    def __init__(self, db):
        self.db = db
        self.current_user = None

    def generate_invite_code(self):
        """Generate unique 6-character invite code"""
        while True:
            code = ''.join(random.choices(string.ascii_uppercase + string.digits, k=6))
            if not self.db.find_class_by_code(code):
                return code

    def hash_password(self, password):
        salt = bcrypt.gensalt()
        return bcrypt.hashpw(password.encode('utf-8'), salt).decode('utf-8')

    def verify_password(self, password, password_hash):
        return bcrypt.checkpw(password.encode('utf-8'), password_hash.encode('utf-8'))

    def register(self, name, email, password, role):
        """Register a new user (teacher or student)"""
        # Check if user exists
        if self.db.find_user_by_email(email):
            return {"success": False, "message": "Email already registered"}

        # Create user
        user_id = str(uuid.uuid4())
        user = {
            'id': user_id,
            'name': name,
            'email': email,
            'password_hash': self.hash_password(password),
            'role': role,
            'created_at': datetime.now().isoformat(),
            'profile_pic': f"https://ui-avatars.com/api/?name={name.replace(' ', '+')}&background=random&color=fff"
        }

        self.db.save_user(user)
        return {"success": True, "message": f"Registration successful! You can now login as {role}", "user_id": user_id}

    def login(self, email, password):
        """Login user"""
        user = self.db.find_user_by_email(email)
        if not user:
            return {"success": False, "message": "User not found"}

        if not self.verify_password(password, user['password_hash']):
            return {"success": False, "message": "Invalid password"}

        self.current_user = user
        return {"success": True, "message": f"Welcome back, {user['name']}!", "user": user}

    def logout(self):
        """Logout current user"""
        self.current_user = None
        return {"success": True, "message": "Logged out successfully"}

    def create_class(self, name, description, subject="", section="", room=""):
        """Create a new class (Teacher only)"""
        if not self.current_user or self.current_user['role'] != 'teacher':
            return {"success": False, "message": "Only teachers can create classes"}

        class_id = str(uuid.uuid4())
        classroom = {
            'id': class_id,
            'name': name,
            'description': description,
            'teacher_id': self.current_user['id'],
            'teacher_name': self.current_user['name'],
            'invite_code': self.generate_invite_code(),
            'created_at': datetime.now().isoformat(),
            'subject': subject,
            'section': section,
            'room': room,
            'total_students': 0
        }

        self.db.save_class(classroom)

        # Auto-enroll teacher
        enrollment = {
            'id': str(uuid.uuid4()),
            'user_id': self.current_user['id'],
            'class_id': class_id,
            'role': 'teacher',
            'enrolled_at': datetime.now().isoformat()
        }
        self.db.add_enrollment(enrollment)

        return {"success": True, "message": f"Class '{name}' created successfully!", "class": classroom}

    def join_class(self, invite_code):
        """Join a class using invite code"""
        if not self.current_user:
            return {"success": False, "message": "Please login first"}

        classroom = self.db.find_class_by_code(invite_code)
        if not classroom:
            return {"success": False, "message": "Invalid invite code"}

        # Check if already enrolled
        if self.db.is_enrolled(self.current_user['id'], classroom['id']):
            return {"success": False, "message": "You are already enrolled in this class"}

        # Add enrollment
        enrollment = {
            'id': str(uuid.uuid4()),
            'user_id': self.current_user['id'],
            'class_id': classroom['id'],
            'role': self.current_user['role'],
            'enrolled_at': datetime.now().isoformat()
        }

        self.db.add_enrollment(enrollment)
        return {"success": True, "message": f"Successfully joined {classroom['name']}!", "class": classroom}

    def get_my_classes(self):
        """Get all classes for current user"""
        if not self.current_user:
            return []

        if self.current_user['role'] == 'teacher':
            classes = self.db.get_teacher_classes(self.current_user['id'])
        else:
            classes = self.db.get_student_classes(self.current_user['id'])

        # Add student count for each class
        for cls in classes:
            enrollments = self.db.get_enrollments()
            student_count = len([e for e in enrollments if e['class_id'] == cls['id'] and e['role'] == 'student'])
            cls['student_count'] = student_count

        return classes

    def create_assignment(self, class_id, title, description, due_date_str, total_points):
        """Create an assignment (Teacher only)"""
        if not self.current_user or self.current_user['role'] != 'teacher':
            return {"success": False, "message": "Only teachers can create assignments"}

        # Verify teacher owns the class
        classroom = self.db.find_class_by_id(class_id)
        if not classroom or classroom['teacher_id'] != self.current_user['id']:
            return {"success": False, "message": "You don't have permission for this class"}

        assignment = {
            'id': str(uuid.uuid4()),
            'class_id': class_id,
            'title': title,
            'description': description,
            'due_date': due_date_str,
            'total_points': total_points,
            'created_at': datetime.now().isoformat(),
            'submissions_count': 0
        }

        self.db.save_assignment(assignment)
        return {"success": True, "message": f"Assignment '{title}' created!", "assignment": assignment}

    def submit_assignment(self, assignment_id, content):
        """Submit an assignment (Student only)"""
        if not self.current_user or self.current_user['role'] != 'student':
            return {"success": False, "message": "Only students can submit assignments"}

        # Check if already submitted
        existing = self.db.get_student_submission(assignment_id, self.current_user['id'])
        if existing:
            return {"success": False, "message": "You have already submitted this assignment. Contact your teacher to resubmit."}

        submission = {
            'id': str(uuid.uuid4()),
            'assignment_id': assignment_id,
            'student_id': self.current_user['id'],
            'student_name': self.current_user['name'],
            'content': content,
            'submitted_at': datetime.now().isoformat(),
            'status': 'submitted'
        }

        self.db.add_submission(submission)
        return {"success": True, "message": "Assignment submitted successfully!"}

    def get_assignments(self, class_id):
        """Get all assignments for a class"""
        assignments = self.db.get_class_assignments(class_id)

        # Add submission status for students
        if self.current_user and self.current_user['role'] == 'student':
            for assignment in assignments:
                submission = self.db.get_student_submission(assignment['id'], self.current_user['id'])
                assignment['submitted'] = submission is not None
                if submission:
                    assignment['submitted_at'] = submission['submitted_at']

        return assignments

    def add_announcement(self, class_id, title, content):
        """Add an announcement (Teacher only)"""
        if not self.current_user or self.current_user['role'] != 'teacher':
            return {"success": False, "message": "Only teachers can post announcements"}

        announcement = {
            'id': str(uuid.uuid4()),
            'class_id': class_id,
            'title': title,
            'content': content,
            'teacher_id': self.current_user['id'],
            'teacher_name': self.current_user['name'],
            'created_at': datetime.now().isoformat()
        }

        self.db.add_announcement(announcement)
        return {"success": True, "message": "Announcement posted!"}

    def get_announcements(self, class_id):
        """Get all announcements for a class"""
        return self.db.get_class_announcements(class_id)

    def grade_submission(self, assignment_id, student_id, points_earned, feedback):
        """Grade a submission (Teacher only)"""
        if not self.current_user or self.current_user['role'] != 'teacher':
            return {"success": False, "message": "Only teachers can grade"}

        # Get assignment to check total points
        assignments = self.db.get_assignments()
        assignment = None
        for a in assignments:
            if a['id'] == assignment_id:
                assignment = a
                break

        if not assignment:
            return {"success": False, "message": "Assignment not found"}

        if points_earned > assignment['total_points']:
            return {"success": False, "message": f"Points cannot exceed {assignment['total_points']}"}

        grade = {
            'assignment_id': assignment_id,
            'student_id': student_id,
            'points_earned': points_earned,
            'feedback': feedback,
            'graded_at': datetime.now().isoformat(),
            'graded_by': self.current_user['name']
        }

        self.db.save_grade(grade)
        return {"success": True, "message": f"Grade submitted! Student earned {points_earned}/{assignment['total_points']} points"}

    def get_student_grade(self, assignment_id, student_id):
        """Get grade for a specific submission"""
        grades = self.db.get_grades()
        for grade in grades:
            if grade['assignment_id'] == assignment_id and grade['student_id'] == student_id:
                return grade
        return None

    def get_class_students(self, class_id):
        """Get all students in a class (Teacher only)"""
        enrollments = self.db.get_enrollments()
        students = []
        for e in enrollments:
            if e['class_id'] == class_id and e['role'] == 'student':
                user = self.db.find_user_by_id(e['user_id'])
                if user:
                    students.append(user)
        return students

    def generate_qr_code(self, class_id):
        """Generate QR code for class invite"""
        classroom = self.db.find_class_by_id(class_id)
        if not classroom:
            return None

        # Create QR code with invite code
        qr = qrcode.QRCode(version=1, box_size=10, border=5)
        qr.add_data(f"CLASS_INVITE:{classroom['invite_code']}")
        qr.make(fit=True)

        qr_image = qr.make_image(fill_color="black", back_color="white")
        return qr_image

# Initialize the manager
manager = ClassroomManager(db)
print("✅ Classroom Manager initialized!")
print("\n📚 System Ready! Continue to next cells to create the interface.")

✅ Classroom Manager initialized!

📚 System Ready! Continue to next cells to create the interface.


In [ ]:
# CELL 5: Create Gradio Web Interface

import gradio as gr
from datetime import datetime

# Global variables to track state
current_view = "login"  # login, register, dashboard

def create_login_ui():
    """Create login interface"""
    with gr.Blocks(theme=gr.themes.Soft()) as login_interface:
        gr.Markdown("""
        # 🎓 Welcome to Google Classroom Clone
        ### Your Complete Learning Management System

        ---
        """)

        with gr.Row():
            with gr.Column(scale=1):
                pass
            with gr.Column(scale=2):
                gr.Markdown("### 🔐 Login to Your Account")
                email = gr.Textbox(label="Email", placeholder="teacher@example.com")
                password = gr.Textbox(label="Password", type="password", placeholder="Enter your password")
                login_btn = gr.Button("Login", variant="primary")
                gr.Markdown("---")
                gr.Markdown("### 🆕 New User?")
                register_btn = gr.Button("Create New Account", variant="secondary")

                login_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=1):
            pass

    return login_interface, login_btn, register_btn, email, password, login_output

def create_register_ui():
    """Create registration interface"""
    with gr.Blocks() as register_interface:
        gr.Markdown("# 📝 Create New Account")

        with gr.Row():
            with gr.Column(scale=1):
                pass
            with gr.Column(scale=2):
                name = gr.Textbox(label="Full Name", placeholder="John Doe")
                email = gr.Textbox(label="Email", placeholder="you@example.com")
                password = gr.Textbox(label="Password", type="password", placeholder="Choose a strong password")
                confirm_password = gr.Textbox(label="Confirm Password", type="password", placeholder="Confirm your password")
                role = gr.Radio(choices=["student", "teacher"], label="I am a:", value="student")
                register_btn = gr.Button("Register", variant="primary")
                back_btn = gr.Button("← Back to Login", variant="secondary")
                register_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=1):
            pass

    return register_interface, register_btn, back_btn, name, email, password, confirm_password, role, register_output

def create_teacher_dashboard():
    """Create teacher dashboard"""
    with gr.Blocks() as teacher_dashboard:
        gr.Markdown("# 👨‍🏫 Teacher Dashboard")

        with gr.Tabs():
            with gr.TabItem("📚 My Classes"):
                refresh_classes_btn = gr.Button("🔄 Refresh Classes")
                classes_list = gr.Dataframe(
                    headers=["Class Name", "Subject", "Section", "Invite Code", "Students"],
                    label="Your Classes"
                )

                gr.Markdown("### ➕ Create New Class")
                with gr.Row():
                    class_name = gr.Textbox(label="Class Name", placeholder="e.g., Computer Science 101")
                    class_subject = gr.Textbox(label="Subject", placeholder="e.g., Programming")
                with gr.Row():
                    class_section = gr.Textbox(label="Section", placeholder="e.g., Section A")
                    class_room = gr.Textbox(label="Room", placeholder="e.g., Room 201")
                class_desc = gr.Textbox(label="Description", placeholder="Class description...", lines=3)
                create_class_btn = gr.Button("Create Class", variant="primary")
                create_output = gr.Textbox(label="Status")

            with gr.TabItem("📝 Manage Assignments"):
                select_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                refresh_assignments_btn = gr.Button("Refresh")

                gr.Markdown("### ✏️ Create New Assignment")
                with gr.Row():
                    assign_title = gr.Textbox(label="Assignment Title", placeholder="Homework 1")
                    assign_points = gr.Number(label="Total Points", value=100)
                assign_desc = gr.Textbox(label="Description", lines=3)
                assign_due = gr.Textbox(label="Due Date", placeholder="YYYY-MM-DD HH:MM")
                create_assign_btn = gr.Button("Create Assignment", variant="primary")
                assign_output = gr.Textbox(label="Status")

                gr.Markdown("### 📊 View Submissions")
                select_assignment = gr.Dropdown(label="Select Assignment", choices=[], interactive=True)
                view_submissions_btn = gr.Button("View Submissions")
                submissions_table = gr.Dataframe(label="Student Submissions")

            with gr.TabItem("📢 Announcements"):
                announce_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                announce_title = gr.Textbox(label="Title", placeholder="Important Announcement")
                announce_content = gr.Textbox(label="Content", lines=5)
                post_announce_btn = gr.Button("Post Announcement", variant="primary")
                announce_output = gr.Textbox(label="Status")

            with gr.TabItem("👥 Students"):
                student_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                view_students_btn = gr.Button("View Students")
                students_table = gr.Dataframe(label="Enrolled Students")

            with gr.TabItem("⚙️ Account"):
                logout_btn = gr.Button("Logout", variant="stop")

        # Add logout functionality
        logout_btn.click(fn=lambda: "logged_out", outputs=[], queue=False)

    return teacher_dashboard

def create_student_dashboard():
    """Create student dashboard"""
    with gr.Blocks() as student_dashboard:
        gr.Markdown("# 🧑‍🎓 Student Dashboard")

        with gr.Tabs():
            with gr.TabItem("📚 My Classes"):
                refresh_student_classes = gr.Button("🔄 Refresh")
                student_classes = gr.Dataframe(
                    headers=["Class Name", "Teacher", "Subject", "Section", "Join Date"],
                    label="Your Classes"
                )

                gr.Markdown("### 🔑 Join a Class")
                invite_code = gr.Textbox(label="Enter Invite Code", placeholder="e.g., ABC123", max_lines=1)
                join_class_btn = gr.Button("Join Class", variant="primary")
                join_output = gr.Textbox(label="Status")

            with gr.TabItem("📝 Assignments"):
                student_select_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                refresh_assignments = gr.Button("Refresh")
                assignments_list = gr.Dataframe(
                    headers=["Assignment", "Due Date", "Points", "Status", "Submitted On", "Grade"],
                    label="Your Assignments"
                )

                gr.Markdown("### 📤 Submit Assignment")
                submit_select = gr.Dropdown(label="Select Assignment", choices=[], interactive=True)
                submission_content = gr.Textbox(label="Your Answer/Work", lines=5)
                submit_btn = gr.Button("Submit Assignment", variant="primary")
                submit_output = gr.Textbox(label="Status")

            with gr.TabItem("📢 Announcements"):
                announce_select_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                view_announcements = gr.Button("View Announcements")
                announcements_display = gr.Markdown("")

            with gr.TabItem("📊 My Grades"):
                grades_select_class = gr.Dropdown(label="Select Class", choices=[], interactive=True)
                view_grades = gr.Button("View Grades")
                grades_display = gr.Dataframe(label="Grades")

            with gr.TabItem("⚙️ Account"):
                student_logout = gr.Button("Logout", variant="stop")

    return student_dashboard

# Create the main app
def create_app():
    with gr.Blocks(theme=gr.themes.Soft(), title="Google Classroom Clone") as app:
        gr.Markdown("""
        # 🎓 Google Classroom Clone

        ### A Complete Learning Management System with Class Codes, Assignments, and Grading

        ---
        """)

        # State management
        current_user = gr.State(None)
        current_role = gr.State(None)

        # Login section
        with gr.Group(visible=True) as login_section:
            gr.Markdown("### 🔐 Login")
            login_email = gr.Textbox(label="Email", placeholder="Enter your email")
            login_password = gr.Textbox(label="Password", type="password", placeholder="Enter password")
            login_btn = gr.Button("Login", variant="primary")
            login_status = gr.Textbox(label="Status", interactive=False)
            goto_register = gr.Button("Don't have an account? Register here", variant="secondary")

        # Register section
        with gr.Group(visible=False) as register_section:
            gr.Markdown("### 📝 Register New Account")
            reg_name = gr.Textbox(label="Full Name")
            reg_email = gr.Textbox(label="Email")
            reg_password = gr.Textbox(label="Password", type="password")
            reg_confirm = gr.Textbox(label="Confirm Password", type="password")
            reg_role = gr.Radio(choices=["student", "teacher"], label="Role", value="student")
            register_btn = gr.Button("Register", variant="primary")
            reg_status = gr.Textbox(label="Status", interactive=False)
            back_to_login = gr.Button("← Back to Login", variant="secondary")

        # Dashboard containers
        teacher_dash = gr.Group(visible=False)
        student_dash = gr.Group(visible=False)

        # Login logic
        def do_login(email, password):
            result = manager.login(email, password)
            if result['success']:
                role = manager.current_user['role']
                return result['message'], role, gr.update(visible=False), gr.update(visible=False), gr.update(visible=True) if role == 'teacher' else gr.update(visible=False), gr.update(visible=True) if role == 'student' else gr.update(visible=False)
            else:
                return result['message'], None, gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

        login_btn.click(
            do_login,
            inputs=[login_email, login_password],
            outputs=[login_status, current_role, login_section, register_section, teacher_dash, student_dash]
        )

        # Registration logic
        def do_register(name, email, password, confirm, role):
            if password != confirm:
                return "Passwords do not match!", gr.update(visible=True), gr.update(visible=False)
            result = manager.register(name, email, password, role)
            if result['success']:
                return result['message'], gr.update(visible=False), gr.update(visible=True)
            return result['message'], gr.update(visible=True), gr.update(visible=False)

        register_btn.click(
            do_register,
            inputs=[reg_name, reg_email, reg_password, reg_confirm, reg_role],
            outputs=[reg_status, register_section, login_section]
        )

        # Navigation
        goto_register.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[login_section, register_section])
        back_to_login.click(lambda: (gr.update(visible=True), gr.update(visible=False)), outputs=[login_section, register_section])

        # Simple dashboard placeholders (you can expand these)
        with teacher_dash:
            gr.Markdown("### 👨‍🏫 Teacher Dashboard")
            gr.Markdown("✅ Login successful! You can now:")
            gr.Markdown("""
            - Create classes
            - Post assignments
            - Grade submissions
            - Make announcements
            """)
            logout_teacher = gr.Button("Logout")

            def teacher_logout():
                manager.logout()
                return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

            logout_teacher.click(teacher_logout, outputs=[login_section, teacher_dash, student_dash])

        with student_dash:
            gr.Markdown("### 🧑‍🎓 Student Dashboard")
            gr.Markdown("✅ Login successful! You can now:")
            gr.Markdown("""
            - Join classes with codes
            - View assignments
            - Submit work
            - Check grades
            """)
            logout_student = gr.Button("Logout")

            def student_logout():
                manager.logout()
                return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

            logout_student.click(student_logout, outputs=[login_section, teacher_dash, student_dash])

    return app

# Launch the app
app = create_app()
app.launch(share=True, debug=True)

print("""
╔═══════════════════════════════════════════════════════════════╗
║          🎓 Google Classroom Clone is Running! 🎓             ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  📝 Quick Start Guide:                                       ║
║                                                               ║
║  1. Register as a TEACHER first                              ║
║  2. Create your first class                                  ║
║  3. Share the invite code with students                      ║
║  4. Register as STUDENT (different email)                    ║
║  5. Join class using invite code                             ║
║  6. Create assignments and submit!                           ║
║                                                               ║
║  🔗 Share the Gradio link with your students                 ║
║  💾 All data is saved to your Google Drive                   ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

/tmp/ipykernel_2573/2334513842.py:172: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Google Classroom Clone") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a8d57c8e18eb886fad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# CELL 6: Admin Dashboard with Analytics

import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML

def show_analytics():
    """Display system analytics"""
    print("\n" + "="*60)
    print("📊 SYSTEM ANALYTICS DASHBOARD")
    print("="*60)

    users = db.get_users()
    classes = db.get_classes()
    assignments = db.get_assignments()
    submissions = db.get_submissions()
    grades = db.get_grades()

    # Statistics
    print(f"\n📈 OVERALL STATISTICS:")
    print(f"   • Total Users: {len(users)}")
    print(f"   • Teachers: {len([u for u in users if u['role'] == 'teacher'])}")
    print(f"   • Students: {len([u for u in users if u['role'] == 'student'])}")
    print(f"   • Total Classes: {len(classes)}")
    print(f"   • Total Assignments: {len(assignments)}")
    print(f"   • Submissions Received: {len(submissions)}")
    print(f"   • Grades Given: {len(grades)}")

    # Calculate average grade
    if grades:
        avg_grade = sum([g['points_earned'] for g in grades]) / len(grades)
        print(f"   • Average Grade: {avg_grade:.1f} points")

    # Class-wise statistics
    print(f"\n📚 CLASS BREAKDOWN:")
    for cls in classes:
        enrollments = db.get_enrollments()
        student_count = len([e for e in enrollments if e['class_id'] == cls['id'] and e['role'] == 'student'])
        class_assignments = len([a for a in assignments if a['class_id'] == cls['id']])
        class_submissions = len([s for s in submissions if s['assignment_id'] in [a['id'] for a in class_assignments]])

        print(f"\n   📖 {cls['name']} ({cls['invite_code']})")
        print(f"      - Students: {student_count}")
        print(f"      - Assignments: {class_assignments}")
        print(f"      - Submissions: {class_submissions}")
        if class_assignments > 0:
            submission_rate = (class_submissions / (student_count * class_assignments)) * 100 if student_count > 0 else 0
            print(f"      - Submission Rate: {submission_rate:.1f}%")

    # Create visualizations
    if classes:
        # Classes over time
        class_dates = [datetime.fromisoformat(c['created_at']) for c in classes]
        class_counts = range(1, len(class_dates) + 1)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=class_dates,
            y=class_counts,
            mode='lines+markers',
            name='Classes Created'
        ))
        fig.update_layout(
            title='Class Creation Timeline',
            xaxis_title='Date',
            yaxis_title='Total Classes',
            template='plotly_dark'
        )
        fig.show()

    # User activity
    if users:
        user_roles = [u['role'] for u in users]
        role_counts = pd.Series(user_roles).value_counts()

        fig = px.pie(
            values=role_counts.values,
            names=role_counts.index,
            title='User Distribution',
            color_discrete_sequence=px.colors.qualitative.Set3
        )
        fig.show()

# Run analytics
show_analytics()

# Quick test commands
print("\n
